# Notebook 1: Path Signatures — A Primer

This notebook introduces the mathematical concept of **path signatures** and shows
how to compute them on financial time series using `pathsig-finance`.

## Contents
1. What is a path signature?
2. Computing signatures on toy paths
3. The Chen identity — building signatures incrementally
4. Signature of a financial time series (GBM)
5. Rolling signatures and the feature dimension


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pathsig.signatures import (
    compute_signature,
    compute_signature_rolling,
    signature_dimension,
    signature_to_dict,
    extract_signature_term,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('pathsig-finance loaded successfully.')

## 1. What is a path signature?

For a path $X: [0,T] \to \mathbb{R}^d$, the **truncated signature** up to depth $N$ is:

$$S(X)^N = \left(1,\; S^1,\; S^2,\; \ldots,\; S^N\right)$$

where the level-$k$ terms are iterated integrals:

$$S(X)^k_{i_1, \ldots, i_k} = \int_{0 < t_1 < \cdots < t_k < T}
  dX^{i_1}_{t_1} \otimes \cdots \otimes dX^{i_k}_{t_k}$$

**Key properties:**
- *Universality*: any continuous function of the path can be approximated by a linear functional of its signature (Lyons, 1998)
- *Invariance*: the signature is invariant to time reparameterisation (same path traversed at different speeds → same signature), which is broken by time-augmentation
- *Chen identity*: $S(X|_{[s,t]}) = S(X|_{[s,u]}) \otimes S(X|_{[u,t]})$ for $s < u < t$

The output dimension is $\sum_{k=1}^N d^k$, which grows with depth but is fixed for a given $(d, N)$.

In [ ]:
# --- Signature dimension ---
print('Signature dimensions (non-constant part):')
print(f'  d=2, depth=1: {signature_dimension(2,1)} (= 2^1)')
print(f'  d=2, depth=2: {signature_dimension(2,2)} (= 2^1 + 2^2)')
print(f'  d=2, depth=3: {signature_dimension(2,3)} (= 2+4+8)')
print(f'  d=3, depth=3: {signature_dimension(3,3)} (= 3+9+27)')
print(f'  d=5, depth=3: {signature_dimension(5,3)} (= 5+25+125)')

## 2. Computing signatures on toy paths

### Straight line in $\mathbb{R}^2$
For a path along a straight line from $(0,0)$ to $(a, b)$:
- Level 1: $S^1 = (a, b)$ — total displacement
- Level 2: $S^2_{ij} = a_i a_j / 2$ — outer product of displacement divided by 2

In [ ]:
# Straight line from (0,0) to (2, 3)
a, b = 2.0, 3.0
line = np.array([[0., 0.], [a, b]])
sig = compute_signature(line, depth=3)
d = signature_to_dict(sig, d=2, depth=3, channel_names=['x', 'y'])

print('Straight line (0,0) → (2,3)')
print(f'  Level 1: S_x={d["S_x"]:.4f} (should be {a}), S_y={d["S_y"]:.4f} (should be {b})')
print(f'  Level 2: S_xx={d["S_xx"]:.4f} (= {a}²/2 = {a**2/2})')
print(f'           S_xy={d["S_xy"]:.4f} (= {a}×{b}/2 = {a*b/2})')
print(f'  Level 3: S_xxx={d["S_xxx"]:.4f} (= {a}³/6 = {a**3/6:.4f})')

In [ ]:
# Visualise several 2-D paths and their level-2 signatures
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

rng = np.random.default_rng(0)
paths = [
    ('Straight line', np.array([[0., 0.], [1., 1.]])),
    ('L-shape', np.array([[0., 0.], [1., 0.], [1., 1.]])),
    ('Zigzag', np.vstack([[0,0]] + [[i*0.2, ((-1)**i)*0.5] for i in range(1,6)])),
]

for ax, (title, path) in zip(axes, paths):
    ax.plot(path[:, 0], path[:, 1], 'b-o', markersize=5)
    ax.plot(path[0, 0], path[0, 1], 'gs', markersize=8, label='start')
    ax.plot(path[-1, 0], path[-1, 1], 'r^', markersize=8, label='end')
    sig = compute_signature(path, depth=2)
    d_dict = signature_to_dict(sig, d=2, depth=2, channel_names=['x', 'y'])
    ax.set_title(f'{title}\nS¹=(%.2f,%.2f)  A_xy=%.3f' % (
        d_dict['S_x'], d_dict['S_y'],
        d_dict['S_xy'] - d_dict['S_yx']  # Lévy area
    ))
    ax.legend(fontsize=8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('2-D Paths and Their Signatures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. The Chen Identity

The **Chen identity** is the fundamental computational tool:
$$S(X|_{[0,T]}) = S(X|_{[0,T/2]}) \otimes S(X|_{[T/2,T]})$$

This lets us build up the signature of a long path by concatenating signatures
of sub-paths — the basis of the incremental algorithm in `pathsig`.

In [ ]:
# Verify Chen identity numerically
rng = np.random.default_rng(1)
path = np.cumsum(rng.standard_normal((20, 3)) * 0.1, axis=0)

split = 10
sig_full = compute_signature(path, depth=3)
sig_left = compute_signature(path[:split+1], depth=3)
sig_right = compute_signature(path[split:], depth=3)

# Manually apply Chen product at level 2 for the first coordinate
# (S_full)^2_{01} = (S_L)^2_{01} + (S_L)^1_0 * (S_R)^1_1 + (S_R)^2_{01}
d = 3
s2_01_left  = sig_left[d + 0*d + 1]   # S^2_{0,1} of left half
s1_0_left   = sig_left[0]              # S^1_0 of left half  
s1_1_right  = sig_right[1]             # S^1_1 of right half
s2_01_right = sig_right[d + 0*d + 1]  # S^2_{0,1} of right half

chen_combined = s2_01_left + s1_0_left * s1_1_right + s2_01_right
sig_full_01   = sig_full[d + 0*d + 1]

print(f'Chen identity verification for S²_{{0,1}}:')
print(f'  From full path:          {sig_full_01:.10f}')
print(f'  From Chen composition:   {chen_combined:.10f}')
print(f'  Absolute difference:     {abs(sig_full_01 - chen_combined):.2e}')

## 4. Signature of a Financial Time Series

In [ ]:
import sys
sys.path.insert(0, '..')
from data.synthetic import generate_gbm_with_leverage

# Generate a single GBM path with leverage
panel = generate_gbm_with_leverage(n_paths=1, n_steps=252, leverage_corr=-0.7, seed=42)
single = panel[panel['ticker'] == 'SYN_0000'].sort_values('date')

# Build the 3-channel path: [log_return, realized_vol, log_volume]
channels = ['log_return', 'realized_vol', 'log_volume']
path_data = single[channels].values  # shape (252, 3)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, ch in zip(axes, channels):
    ax.plot(single['date'].values, single[ch].values)
    ax.set_ylabel(ch, fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Date')
plt.suptitle('Synthetic GBM with Leverage Effect (ρ = −0.7)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from pathsig.augmentations import time_augmentation, lead_lag_augmentation

window = 21
path_window = path_data[:window]  # first 21 days

# Apply time + lead-lag augmentation
aug = lead_lag_augmentation(time_augmentation(path_window))
print(f'Original path shape:   {path_window.shape}')
print(f'After time aug:        {time_augmentation(path_window).shape}')
print(f'After time+lead-lag:   {aug.shape}')

# Compute signature
sig = compute_signature(aug, depth=2)
d_aug = aug.shape[1]
print(f'\nSignature dimension (d={d_aug}, depth=2): {len(sig)}')
print(f'First 10 signature terms: {sig[:10].round(6)}')

## 5. Rolling Signatures and the Feature Matrix

In [ ]:
from pathsig.signatures import compute_signature_rolling
from pathsig.augmentations import apply_augmentations

window = 21
depth  = 2

# Apply augmentations to the full time series
aug_full = apply_augmentations(path_data, ['time', 'lead_lag'])
print(f'Full augmented path shape: {aug_full.shape}')

# Rolling signatures
roll_sigs = compute_signature_rolling(aug_full, window=window, depth=depth)
print(f'Rolling signature matrix: {roll_sigs.shape}')
print(f'  → {roll_sigs.shape[0]} windows × {roll_sigs.shape[1]} features')

# Plot the first 6 signature features over time
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
axes = axes.ravel()
for i in range(6):
    axes[i].plot(roll_sigs[:, i])
    axes[i].set_title(f'Signature feature #{i+1}')
    axes[i].grid(True, alpha=0.3)
plt.suptitle(f'Rolling Signature Features (window={window}, depth={depth})',
             fontweight='bold')
plt.tight_layout()
plt.show()